# Gold ML-features 

In [0]:
%run ../config/config

In [0]:
from pyspark.sql import functions as F

#### 1. Read & join all feature tables

In [0]:
cleaned_df = spark.table(tables["cleaned_ohlcv"])
indicators_df = spark.table(tables["technical_indicators"])
wyckoff_df = spark.table(tables["wyckoff_features"])
labels_df = spark.table(tables["phase_labels"])

full_df = (
    cleaned_df
    .filter((F.col("date") >= train_start) & (F.col("date") <= data_end))
    .join(indicators_df, on=["symbol", "date"], how="inner")
    .join(wyckoff_df, on=["symbol", "date"], how="inner")
    .join(
        labels_df.select("symbol", "date", "phase_label", "phase_confidence"),
        on=["symbol", "date"],
        how="inner",
    )
).cache()

print(f"Full rows: {full_df.count():,}")
print(f"Columns: {len(full_df.columns)}")


#### 2. Prepare feature columns

In [0]:
feature_cols = [c for c in ml_feature_columns if c in full_df.columns]

missing = [c for c in ml_feature_columns if c not in full_df.columns]
if missing:
    print(f"Miss feature columns: {missing}")

# cast booleans to double
cast_exprs = []
for col_name in full_df.columns:
    if col_name in boolean_feature_columns:
        cast_exprs.append(F.col(col_name).cast("double").alias(col_name))
    else:
        cast_exprs.append(F.col(col_name))
full_df = full_df.select(*cast_exprs)

# add temporal split
full_df = full_df.withColumn(
    "split",
    F.when(F.col("date") <= F.lit(train_end), F.lit("train"))
    .when(F.col("date") <= F.lit(val_end), F.lit("val"))
    .otherwise(F.lit("test"))
)

pl_rate = full_df.select(F.mean("price_limit_hit")).collect()[0][0]
print(f"Feature columns: {len(feature_cols)}")
print(f"Price-limit-hit rate: {pl_rate:.4f} ({pl_rate*100:.1f}%)")
display(full_df.groupBy("split").count().orderBy("split"))

#### 3. Select & write flat rows


In [0]:
select_cols = ["symbol", "date", "exchange", "split", "phase_label", "price_limit_hit", "daily_return"] + feature_cols
select_cols = list(dict.fromkeys(select_cols)) 
output_df = full_df.select(*[c for c in select_cols if c in full_df.columns])

full_df.unpersist()

# check if per-symbol row counts
sym_stats = output_df.groupBy("symbol").agg(F.count("*").alias("n"))
sym_agg = sym_stats.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("n") < window_size, 1).otherwise(0)).alias("short"),
).collect()[0]
print(f"Symbols: {sym_agg['total']}, with < {window_size} rows: {sym_agg['short']}")

In [0]:
row_count = output_df.count()
output_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tables["ml_features"])
print(f"Successful written {row_count:,} flat rows to {tables['ml_features']}")

#### 4. Verify

In [0]:
%sql
SELECT
    split,
    COUNT(*) AS window_count
FROM wyckoff_catalog.gold.ml_features
GROUP BY split
ORDER BY split;

In [0]:
%sql
SELECT * 
FROM wyckoff_catalog.gold.ml_features

In [0]:
%sql
SELECT
    phase_label,
    COUNT(*) AS count
FROM wyckoff_catalog.gold.ml_features
GROUP BY phase_label
ORDER BY phase_label

In [0]:
sample_sym = spark.table(tables["ml_features"]).select("symbol").distinct().limit(1).collect()[0]["symbol"]
sample_df = spark.table(tables["ml_features"]).filter(F.col("symbol") == sample_sym).orderBy("date")
print(f"Sample symbol: {sample_sym}, rows: {sample_df.count()}")
print(f"Date range: {sample_df.select(F.min('date'), F.max('date')).collect()[0]}")
print(f"Columns: {len(sample_df.columns)}")